# spaCy Fine-Tuning — Political & Economic NER

## Overview
Fine-tuning unui model spaCy + `bert-base-uncased` pe acelasi dataset politic/economic,
cu scopul compararii directe cu modelul GLiNER antrenat anterior.

**De ce bert-base-uncased ca model de baza comun?**
- Acelasi backbone transformer ca si GLiNER medium (DeBERTa-v3 e mai puternic, dar
  pentru comparatie corecta folosim acelasi tip de model)
- `dslim/bert-base-NER` — BERT pretrained deja pe CoNLL-2003 NER → punct de start mai bun
  decat BERT vanilla pentru task-ul nostru

**Diferenta arhitecturala fata de GLiNER (recapitulare):**
```
spaCy:   Text → Transformer → Linear Head (fix, IOB2) → entitati per token
GLiNER:  Text + Labels → Transformer → Span Scorer (cosine sim) → entitati per span
```

**Pipeline:**
1. Instalare dependinte
2. Variabile globale (aceleasi cai ca in GLiNER notebook)
3. Conversie date in format spaCy (DocBin)
4. Generare config.cfg pentru transformer NER
5. Incarcare si initializare model
6. Antrenare (in-process, fara CLI)
7. Evaluare (Precision / Recall / F1 per label)
8. Inferenta pe texte noi
9. Salvare model
10. Tabel comparativ GLiNER vs spaCy

---
**Hardware recomandat:** GPU T4 (Colab gratuit)
**Timp estimat:** 20-40 minute pe T4

## PASUL 1 — Instalare dependinte

Ruleaza aceasta celula si **restarteza runtime-ul** dupa finalizare.
Nu re-rula dupa restart.

**Nota:** `spacy-transformers` este extensia care leaga spaCy de HuggingFace transformers.
Fara ea, `[components.transformer]` din config nu functioneaza.

In [ ]:
!pip install -q spacy spacy-transformers torch
!python -m spacy download en_core_web_sm -q
print("Instalare completa. Mergi la Runtime > Restart Runtime, apoi continua.")

## PASUL 2 — Variabile globale (MODIFICA AICI)

**Acelasi `DRIVE_ROOT` ca in notebook-ul GLiNER** — datele sunt deja acolo.

Singurul lucru nou: directoarele de output pentru modelul spaCy.

In [ ]:
# ─── MONTEAZA GOOGLE DRIVE ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# VARIABILE GLOBALE — MODIFICA ACESTE CAI DUPA STRUCTURA TA DE DRIVE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Aceleasi cai ca in GLiNER notebook — datele nu se schimba
DRIVE_ROOT = "/content/drive/MyDrive"   # ← MODIFICA DACA E NECESAR

TRAIN_PATH = f"{DRIVE_ROOT}/dataset/splits/train.jsonl"
DEV_PATH   = f"{DRIVE_ROOT}/dataset/splits/dev.jsonl"
TEST_PATH  = f"{DRIVE_ROOT}/dataset/splits/test.jsonl"

# Modelul transformer de baza
# dslim/bert-base-NER — BERT pretrained pe CoNLL-2003, punct de start mai bun
# Alternativa: "bert-base-uncased" (vanilla, fara pretraining NER)
BASE_MODEL = "dslim/bert-base-NER"

# Directoare output spaCy
SPACY_DATA_DIR    = "/content/spacy_data"          # .spacy DocBin files
SPACY_CONFIG_PATH = "/content/spacy_data/config.cfg"
SPACY_OUTPUT_DIR  = "/content/spacy_output"        # checkpoints antrenare
OUTPUT_MODEL_DIR  = f"{DRIVE_ROOT}/models/spacy_polteco_final"

# Unde sunt salvate rezultatele GLiNER (pentru comparatie finala)
# Lasa None daca nu ai rulat inca GLiNER notebook
GLINER_RESULTS_PATH = f"{DRIVE_ROOT}/models/gliner_polteco_final/training_metadata.json"

# Labelurile NER — IDENTICE cu cele din GLiNER notebook pentru comparatie corecta
NER_LABELS = [
    "POLITICIAN",
    "POLITICAL_PARTY",
    "POLITICAL_ORG",
    "FINANCIAL_ORG",
    "ECONOMIC_INDICATOR",
    "POLICY",
    "LEGISLATION",
    "MARKET_EVENT",
    "CURRENCY",
    "TRADE_AGREEMENT",
    "GPE",
]

# ─── Hyperparametri antrenare ─────────────────────────────────────────────────
# Calibrati pentru comparatie corecta cu GLiNER:
# - Acelasi numar de epoci
# - Acelasi seed
# spaCy foloseste max_steps in loc de epoci — calculam echivalentul

NUM_EPOCHS       = 10
BATCH_SIZE       = 8     # spaCy e mai eficient cu batches mai mari
LEARNING_RATE    = 5e-5  # standard pentru BERT fine-tuning
WEIGHT_DECAY     = 0.01
WARMUP_STEPS     = 250
SEED             = 42
DROPOUT          = 0.1
GRAD_ACCUM_STEPS = 3     # acumuleaza 3 batch-uri inainte de update

# ─── Verificare cai ───────────────────────────────────────────────────────────
import os
from pathlib import Path

for d in [SPACY_DATA_DIR, SPACY_OUTPUT_DIR, OUTPUT_MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print("=== Configurare ===")
for name, path in [("TRAIN", TRAIN_PATH), ("DEV", DEV_PATH), ("TEST", TEST_PATH)]:
    exists = Path(path).exists()
    status = "✅" if exists else "❌ FISIER NEGASIT"
    print(f"  {name}: {path} {status}")

print(f"\n  Base model:    {BASE_MODEL}")
print(f"  Output model:  {OUTPUT_MODEL_DIR}")
print(f"  NER labels:    {NER_LABELS}")

## PASUL 3 — Verificare GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU detectat: {gpu}")
    print(f"VRAM total:   {vram:.1f} GB")
    USE_GPU = 0  # spaCy foloseste index 0
else:
    print("ATENTIE: GPU nu disponibil. Mergi la Runtime > Change runtime type > T4 GPU")
    USE_GPU = -1  # spaCy: -1 = CPU

import spacy
spacy.prefer_gpu(USE_GPU)
print(f"\nspaCy version: {spacy.__version__}")
print(f"GPU index pentru spaCy: {USE_GPU}")

## PASUL 4 — Conversie date in format spaCy (DocBin)

**Diferenta fata de GLiNER:**
- GLiNER folosea: `{"tokenized_text": [...], "ner": [[start_tok, end_tok, label]]}`
- spaCy foloseste: **DocBin** — un format binar cu obiecte `Doc` care contin `doc.ents`

**Conversia** porneste din formatul nostru JSONL (character offsets) direct in DocBin.
Nu mai e nevoie de conversia char→token ca la GLiNER — spaCy face asta intern prin
`doc.char_span(start, end, label=label)`.

**Atentie la boundary misalignment:** `char_span()` returneaza `None` daca span-ul
nu se aliniaza cu limitele tokenilor spaCy. Celula raporteaza cate entitati sunt sarite.

In [ ]:
import json
import spacy
from spacy.tokens import DocBin
from pathlib import Path
from collections import Counter

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

def convert_to_spacy(examples, output_path, split_name=""):
    """
    Converteste din formatul nostru JSONL in format spaCy DocBin.

    Input (formatul nostru):
    {
        "text": "The Fed raised rates",
        "entities": [
            {"text": "Fed", "label": "FINANCIAL_ORG", "start": 4, "end": 7}
        ]
    }

    Output: fisier .spacy (DocBin binar)

    Diferenta fata de GLiNER: char_span() se ocupa de alinierea cu tokenizarea spaCy.
    Nu e nevoie de conversia manuala char→token.
    """
    nlp_blank = spacy.blank("en")
    db        = DocBin()

    skipped_ents  = 0
    skipped_docs  = 0
    label_counts  = Counter()
    skip_per_label = Counter()

    for ex in examples:
        text    = ex["text"]
        doc     = nlp_blank.make_doc(text)
        ents    = []
        doc_ok  = True

        for ent in ex.get("entities", []):
            span = doc.char_span(ent["start"], ent["end"], label=ent["label"])
            if span is None:
                # Incearca cu alignment_mode="expand" — preia tokenul cel mai apropiat
                span = doc.char_span(
                    ent["start"], ent["end"],
                    label=ent["label"],
                    alignment_mode="expand"
                )
            if span is not None:
                ents.append(span)
                label_counts[ent["label"]] += 1
            else:
                skipped_ents += 1
                skip_per_label[ent["label"]] += 1

        # Filtreaza overlap-uri (spaCy nu suporta entitati suprapuse)
        try:
            doc.ents = ents
        except ValueError:
            # Overlap detection — pastreaza cea mai lunga entitate
            filtered = []
            ents_sorted = sorted(ents, key=lambda s: s.end - s.start, reverse=True)
            used = set()
            for span in ents_sorted:
                positions = set(range(span.start, span.end))
                if not positions & used:
                    filtered.append(span)
                    used.update(positions)
            doc.ents = filtered

        db.add(doc)

    db.to_disk(output_path)

    print(f"  [{split_name}] {len(examples)} exemple → {output_path}")
    print(f"    Entitati convertite: {sum(label_counts.values())}")
    print(f"    Entitati sarite:     {skipped_ents}")
    if skip_per_label:
        for label, cnt in skip_per_label.most_common():
            print(f"      Skip {label}: {cnt}")
    return label_counts


# ─── Incarcare si conversie ───────────────────────────────────────────────────
print("Se incarca datele...")
raw_train = load_jsonl(TRAIN_PATH)
raw_dev   = load_jsonl(DEV_PATH)
raw_test  = load_jsonl(TEST_PATH)
print(f"Incarcat: {len(raw_train)} train / {len(raw_dev)} dev / {len(raw_test)} test\n")

print("Se convertesc in format spaCy DocBin...")
train_labels = convert_to_spacy(raw_train, f"{SPACY_DATA_DIR}/train.spacy", "train")
dev_labels   = convert_to_spacy(raw_dev,   f"{SPACY_DATA_DIR}/dev.spacy",   "dev")
test_labels  = convert_to_spacy(raw_test,  f"{SPACY_DATA_DIR}/test.spacy",  "test")

print("\nDistributie labeluri in train (dupa conversie spaCy):")
for label in NER_LABELS:
    cnt = train_labels.get(label, 0)
    bar = "█" * (cnt // 5)
    print(f"  {label:<25} {cnt:>4}  {bar}")

## PASUL 5 — Generare config.cfg

**De ce config.cfg?**
spaCy 3.x foloseste un sistem de configurare declarativ — tot ce priveste arhitectura
modelului, hyperparametrii si datele este specificat intr-un singur fisier `.cfg`.

Generam config-ul programatic (nu prin CLI) si il personalizam cu:
- `bert-base-uncased` sau `dslim/bert-base-NER` ca backbone transformer
- Labelurile noastre NER
- Hyperparametrii definiti in Pasul 2

In [ ]:
# ─── Generare config.cfg pentru transformer NER ──────────────────────────────
# Construim config-ul manual pentru control complet.
# Structura este compatibila cu spaCy 3.x + spacy-transformers.

STEPS_PER_EPOCH  = len(raw_train) // BATCH_SIZE
TOTAL_STEPS      = STEPS_PER_EPOCH * NUM_EPOCHS
EVAL_FREQUENCY   = max(50, STEPS_PER_EPOCH)  # evalueaza o data pe epoca

config_content = f"""
[paths]
train = "{SPACY_DATA_DIR}/train.spacy"
dev   = "{SPACY_DATA_DIR}/dev.spacy"

[system]
gpu_allocator = "pytorch"
seed = {SEED}

[nlp]
lang = "en"
pipeline = ["transformer", "ner"]
batch_size = {BATCH_SIZE}

[components]

[components.transformer]
factory = "transformer"
max_batch_items = 4096

[components.transformer.model]
@architectures = "spacy-transformers.TransformerModel.v3"
name = "{BASE_MODEL}"
mixed_precision = false

[components.transformer.model.get_spans]
@span_getters = "spacy-transformers.strided_spans.v1"
window = 128
stride = 96

[components.transformer.model.tokenizer_config]
use_fast = true

[components.transformer.model.transformer_config]

[components.transformer.model.grad_scaler_config]

[components.transformer.set_extra_annotations]
@annotation_setters = "spacy-transformers.null_annotation_setter.v1"

[components.ner]
factory = "ner"
moves = null
update_with_oracle_cut_size = 100

[components.ner.model]
@architectures = "spacy.TransitionBasedParser.v2"
state_type = "ner"
extra_state_tokens = false
hidden_width = 64
maxout_pieces = 2
use_upper = false
nO = null

[components.ner.model.tok2vec]
@architectures = "spacy-transformers.TransformerListener.v1"
grad_factor = 1.0
upstream = "*"

[components.ner.model.tok2vec.pooling]
@layers = "reduce_mean.v1"

[components.ner.scorer]
@scorers = "spacy.ner_scorer.v1"

[corpora]

[corpora.train]
@readers = "spacy.Corpus.v1"
path = ${{paths.train}}
max_length = 0
gold_preproc = false

[corpora.dev]
@readers = "spacy.Corpus.v1"
path = ${{paths.dev}}
max_length = 0
gold_preproc = false

[training]
train_corpus = "corpora.train"
dev_corpus   = "corpora.dev"
seed         = ${{system.seed}}
gpu_allocator = ${{system.gpu_allocator}}
accumulate_gradient = {GRAD_ACCUM_STEPS}
dropout     = {DROPOUT}
patience    = 3000
max_epochs  = {NUM_EPOCHS}
max_steps   = 0
eval_frequency = {EVAL_FREQUENCY}
frozen_components = []
annotating_components = []
before_to_disk = null
before_update = null

[training.batcher]
@batchers = "spacy.batch_by_words.v1"
discard_oversize = false
tolerance = 0.2
get_length = null

[training.batcher.size]
@schedules = "compounding.v1"
start = {BATCH_SIZE}
stop  = {BATCH_SIZE * 4}
compound = 1.001

[training.logger]
@loggers = "spacy.ConsoleLogger.v1"
progress_bar = true

[training.optimizer]
@optimizers = "Adam.v1"
beta1 = 0.9
beta2 = 0.999
L2_is_weight_decay = true
L2 = {WEIGHT_DECAY}
grad_clip = 1.0
use_averages = false
eps = 1e-08

[training.optimizer.learn_rate]
@schedules = "warmup_linear.v1"
warmup_steps = {WARMUP_STEPS}
total_steps  = {max(TOTAL_STEPS, WARMUP_STEPS + 100)}
initial_rate = {LEARNING_RATE}

[training.score_weights]
ents_f = 1.0
ents_p = 0.0
ents_r = 0.0
ents_per_type = null

[initialize]
vectors = null
init_tok2vec = null
vocab_data = null
lookups = null
before_init = null
after_init = null

[initialize.tokenizer]
[initialize.components]
"""

with open(SPACY_CONFIG_PATH, "w") as f:
    f.write(config_content)

print(f"Config salvat: {SPACY_CONFIG_PATH}")
print(f"  Base model:      {BASE_MODEL}")
print(f"  Epoci:           {NUM_EPOCHS}")
print(f"  Batch size:      {BATCH_SIZE}")
print(f"  Total steps:     {TOTAL_STEPS}")
print(f"  Eval frequency:  {EVAL_FREQUENCY} steps")
print(f"  Learning rate:   {LEARNING_RATE}")
print(f"  Warmup steps:    {WARMUP_STEPS}")

## PASUL 6 — Initializare model

**Ce face initializarea?**
1. Descarca `dslim/bert-base-NER` de pe HuggingFace (~450MB)
2. Adauga labelurile noastre NER in vocabularul modelului
3. Initializeaza head-ul de clasificare (layerul linear de deasupra BERT)

**Diferenta fata de GLiNER:** spaCy trebuie sa stie labelurile NER la initializare
(tagset fix). GLiNER primea labelurile dinamic la inferenta.

In [ ]:
import spacy
from spacy.cli.train import train as spacy_train
from spacy.training.initialize import init_nlp
from spacy import Config

# ─── Incarcare config si initializare ────────────────────────────────────────
print(f"Se incarca config si se initializeaza modelul: {BASE_MODEL}")
print("(Prima rulare descarca ~450MB de pe HuggingFace...)\n")

config     = Config().from_disk(SPACY_CONFIG_PATH)
nlp        = init_nlp(config, use_gpu=USE_GPU)

# ─── Adaugare labeluri NER ────────────────────────────────────────────────────
# spaCy trebuie sa cunoasca toate labelurile INAINTE de antrenare.
# Spre deosebire de GLiNER care primeste labelurile dinamic la fiecare inferenta.
ner = nlp.get_pipe("ner")
for label in NER_LABELS:
    ner.add_label(label)

# Informatii model
total_params = sum(
    p.numel() for p in nlp.get_pipe("transformer").model.shims[0]._model.parameters()
)
print(f"Model initializat pe: {'GPU' if USE_GPU >= 0 else 'CPU'}")
print(f"Parametri transformer: {total_params:,}")
print(f"Labeluri NER adaugate: {ner.labels}")
print(f"Numar labeluri:        {len(ner.labels)}")

# Test rapid pe un text — inainte de fine-tuning (va da rezultate slabe/gresite)
print("\n=== Test INAINTE de fine-tuning (baseline zero) ===")
test_text = "Jerome Powell raised interest rates as the Federal Reserve fights inflation."
doc = nlp(test_text)
if doc.ents:
    for ent in doc.ents:
        print(f"  '{ent.text}' → {ent.label_}")
else:
    print("  Nicio entitate detectata (normal — modelul nu e antrenat inca pe labelurile noastre)")

## PASUL 7 — Antrenare

**Ce se intampla in aceasta celula:**
1. spaCy ruleaza training loop-ul intern (nu expunem manual backward/forward)
2. La fiecare `eval_frequency` steps: evalueaza pe dev set → afiseaza F1
3. `patience = 3000` — daca F1 nu creste timp de 3000 steps, se opreste automat
4. Cel mai bun model (highest F1 pe dev) e salvat in `SPACY_OUTPUT_DIR/model-best`

**Monitorizare:** urmareste coloana `NER F` — F1 pe dev set. Daca stagneaza = underfitting.

**Daca primesti OOM:**
- Reduce `BATCH_SIZE` la 4 in variabilele globale
- Re-ruleaza de la Pasul 5 (regenereaza config)

In [ ]:
import time
from pathlib import Path

# ─── DE CE spacy_train() SI NU training loop manual ──────────────────────────
# spacy.cli.train.train() este API-ul oficial pentru antrenare in spaCy 3.x.
# Alternativa (training loop manual) necesita gestionarea explicita a:
#   - batch-urilor de Example objects
#   - optimizer step
#   - gradient accumulation
#   - evaluate + save best model
# spacy_train() le face pe toate corect si eficient.

print("=" * 60)
print("INCEPE ANTRENAREA")
print("=" * 60)
print(f"Train examples:  {len(raw_train)}")
print(f"Dev examples:    {len(raw_dev)}")
print(f"Epoci maxime:    {NUM_EPOCHS}")
print(f"Metoda:          spacy_train() — loop nativ spaCy")
print("=" * 60)

start_time = time.time()

spacy_train(
    config_path = SPACY_CONFIG_PATH,
    output_path = SPACY_OUTPUT_DIR,
    use_gpu     = USE_GPU,
    overrides   = {},
)

elapsed = time.time() - start_time
print("\n" + "=" * 60)
print("ANTRENARE FINALIZATA")
print("=" * 60)
print(f"Timp total:     {elapsed/60:.1f} minute")
print(f"Model best:     {SPACY_OUTPUT_DIR}/model-best")
print(f"Model last:     {SPACY_OUTPUT_DIR}/model-last")

# Incarcam cel mai bun model pentru evaluare
nlp_best = spacy.load(f"{SPACY_OUTPUT_DIR}/model-best")
print(f"\nModel best incarcat. Labeluri NER: {nlp_best.get_pipe('ner').labels}")

## PASUL 8 — Evaluare pe Test Set

Folosim **aceeasi functie de evaluare** ca in notebook-ul GLiNER pentru comparatie
directa: match exact pe (text, label) — nu contorizeaza partial matches.

**Metrici raportate:**
- **Precision / Recall / F1** — global si per label
- **Support** — numarul de entitati reale per label in test set

In [ ]:
from collections import defaultdict

def evaluate_spacy(nlp_model, test_examples_raw, labels):
    """
    Evalueaza modelul spaCy pe test set.
    Aceeasi logica ca evaluate_gliner() din notebook-ul GLiNER
    pentru comparatie directa.
    """
    tp_global, fp_global, fn_global = 0, 0, 0
    per_label = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

    for ex in test_examples_raw:
        text    = ex["text"]
        gt_ents = ex.get("entities", [])

        gt_set   = {(e["text"].lower(), e["label"]) for e in gt_ents}
        doc      = nlp_model(text)
        pred_set = {(ent.text.lower(), ent.label_) for ent in doc.ents}

        for ent in pred_set:
            if ent in gt_set:
                tp_global += 1
                per_label[ent[1]]["tp"] += 1
            else:
                fp_global += 1
                per_label[ent[1]]["fp"] += 1

        for ent in gt_set:
            if ent not in pred_set:
                fn_global += 1
                per_label[ent[1]]["fn"] += 1

    def prf(tp, fp, fn):
        p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        return p, r, f1

    gp, gr, gf1 = prf(tp_global, fp_global, fn_global)

    label_metrics = {}
    for label in labels:
        c = per_label[label]
        p, r, f1 = prf(c["tp"], c["fp"], c["fn"])
        label_metrics[label] = {
            "precision": p, "recall": r, "f1": f1,
            "support": c["tp"] + c["fn"]
        }

    return {
        "global":    {"precision": gp, "recall": gr, "f1": gf1,
                      "tp": tp_global, "fp": fp_global, "fn": fn_global},
        "per_label": label_metrics
    }


print("Se evalueaza pe test set...")
spacy_results = evaluate_spacy(nlp_best, raw_test, NER_LABELS)

# ─── Afisare rezultate ─────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("REZULTATE EVALUARE — TEST SET (spaCy)")
print("=" * 65)

g = spacy_results["global"]
print(f"\n  GLOBAL:")
print(f"    Precision: {g['precision']:.4f}")
print(f"    Recall:    {g['recall']:.4f}")
print(f"    F1:        {g['f1']:.4f}")
print(f"    TP/FP/FN:  {g['tp']} / {g['fp']} / {g['fn']}")

print(f"\n  {'LABEL':<25} {'P':>6} {'R':>6} {'F1':>6} {'SUPPORT':>8}")
print(f"  {'-'*55}")
for label in NER_LABELS:
    m = spacy_results["per_label"][label]
    if m["support"] > 0:
        flag = "" if m["f1"] >= 0.5 else " ←"
        print(f"  {label:<25} {m['precision']:>6.3f} {m['recall']:>6.3f} {m['f1']:>6.3f} {m['support']:>8}{flag}")
    else:
        print(f"  {label:<25} {'N/A':>6} {'N/A':>6} {'N/A':>6} {0:>8}")

print("\n  ← Labels cu F1 < 0.5")

## PASUL 9 — Inferenta pe texte noi

Testam modelul spaCy pe **aceleasi propozitii** ca in notebook-ul GLiNER
pentru comparatie vizuala directa a predictiilor.

In [ ]:
import json, time

# Aceleasi propozitii de test ca in GLiNER notebook
TEST_SENTENCES = [
    "Jerome Powell signaled that the Federal Reserve may keep interest rates higher for longer to combat persistent inflation.",
    "The European Commission launched an investigation into Goldman Sachs over alleged violations of MiFID II regulations.",
    "The Republican Party blocked the new fiscal stimulus package, citing concerns about the rising budget deficit and GDP growth targets.",
    "NAFTA's successor, the USMCA, imposed stricter rules of origin on automotive manufacturing in Mexico and Canada.",
    "Bitcoin surged past $70,000 after the SEC approved the first spot Bitcoin ETF, drawing massive institutional inflows.",
    "The IMF downgraded its global growth forecast as the 2008 financial crisis continued to reverberate through emerging markets.",
    "Christine Lagarde announced that the ECB would begin quantitative tightening, reducing its bond portfolio by €25 billion monthly.",
    "NATO members committed to spending 2% of GDP on defense following Russia's military escalation in Eastern Europe.",
]

print("=" * 70)
print("INFERENTA MODEL spaCy FINE-TUNAT")
print("=" * 70)

for text in TEST_SENTENCES:
    doc = nlp_best(text)
    print(f"\nText: {text[:80]}..." if len(text) > 80 else f"\nText: {text}")
    if doc.ents:
        for ent in doc.ents:
            print(f"  '{ent.text}' → {ent.label_}")
    else:
        print("  [nicio entitate detectata]")

In [ ]:
# ─── Output JSON structurat (acelasi format ca GLiNER) ───────────────────────
def analyze_text_spacy(text, nlp_model, labels=NER_LABELS):
    """Ruleaza NER spaCy si returneaza JSON structurat — acelasi format ca GLiNER."""
    start = time.time()
    doc   = nlp_model(text)
    elapsed_ms = round((time.time() - start) * 1000)

    return {
        "entities": [
            {
                "text":   ent.text,
                "label":  ent.label_,
                "start":  ent.start_char,
                "end":    ent.end_char,
                "score":  None,   # spaCy nu expune confidence score per entitate
                "source": "spacy_finetuned"
            }
            for ent in sorted(doc.ents, key=lambda e: e.start_char)
            if ent.label_ in labels
        ],
        "metadata": {
            "model":              BASE_MODEL,
            "processing_time_ms": elapsed_ms,
            "char_count":         len(text)
        }
    }

sample = "Jerome Powell raised rates as the Federal Reserve battles inflation amid the USMCA trade tensions."
result = analyze_text_spacy(sample, nlp_best)
print("Output JSON final (format identic cu GLiNER):")
print(json.dumps(result, indent=2))

## PASUL 10 — Salvare model pe Drive

In [ ]:
import os, shutil

print(f"Se salveaza modelul in: {OUTPUT_MODEL_DIR}")
os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)

# Copiem model-best (cel mai bun checkpoint) pe Drive
best_path = Path(f"{SPACY_OUTPUT_DIR}/model-best")
if best_path.exists():
    if Path(OUTPUT_MODEL_DIR).exists():
        shutil.rmtree(OUTPUT_MODEL_DIR)
    shutil.copytree(str(best_path), OUTPUT_MODEL_DIR)
    print("Model-best copiat pe Drive.")
else:
    print("ATENTIE: model-best nu exista. Verificati ca antrenarea s-a finalizat.")

# Salvam metadatele de evaluare (acelasi format ca GLiNER pentru comparatie)
eval_metadata = {
    "base_model":          BASE_MODEL,
    "fine_tuned_on":       "political_economic_ner_dataset",
    "train_examples":      len(raw_train),
    "dev_examples":        len(raw_dev),
    "test_examples":       len(raw_test),
    "ner_labels":          NER_LABELS,
    "hyperparameters": {
        "learning_rate":    LEARNING_RATE,
        "num_epochs":       NUM_EPOCHS,
        "batch_size":       BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "warmup_steps":     WARMUP_STEPS,
        "weight_decay":     WEIGHT_DECAY,
    },
    "evaluation": {
        "global_precision": spacy_results["global"]["precision"],
        "global_recall":    spacy_results["global"]["recall"],
        "global_f1":        spacy_results["global"]["f1"],
        "per_label_f1":     {l: spacy_results["per_label"][l]["f1"] for l in NER_LABELS}
    }
}

import json
with open(os.path.join(OUTPUT_MODEL_DIR, "training_metadata.json"), "w") as f:
    json.dump(eval_metadata, f, indent=2)

print(f"\nMetadata salvat.")
print(f"Global F1: {spacy_results['global']['f1']:.4f}")
print(f"\nPentru a incarca modelul mai tarziu:")
print(f'  nlp = spacy.load("{OUTPUT_MODEL_DIR}")')

## PASUL 11 — Tabel comparativ GLiNER vs spaCy

Aceasta celula incarca rezultatele ambelor modele si genereaza tabelul
de comparatie pentru lucrare.

**Daca nu ai rulat inca GLiNER notebook:** celula va afisa doar rezultatele spaCy
si va lasa coloana GLiNER goala.

In [ ]:
import json
from pathlib import Path

# ─── Incarcare rezultate GLiNER (daca exista) ─────────────────────────────────
gliner_meta = None
if GLINER_RESULTS_PATH and Path(GLINER_RESULTS_PATH).exists():
    with open(GLINER_RESULTS_PATH) as f:
        gliner_meta = json.load(f)
    print(f"Rezultate GLiNER incarcate din: {GLINER_RESULTS_PATH}")
else:
    print("Rezultate GLiNER nu au fost gasite. Se afiseaza doar spaCy.")
    print(f"Cale asteptata: {GLINER_RESULTS_PATH}")

# ─── Tabel comparativ ─────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("COMPARATIE GLINER vs spaCy — Fine-tuned pe acelasi dataset")
print("=" * 80)

# Header
if gliner_meta:
    print(f"\n  {'LABEL':<25} {'GLiNER F1':>10} {'spaCy F1':>10} {'DIFERENTA':>12}")
    print(f"  {'-'*62}")
else:
    print(f"\n  {'LABEL':<25} {'spaCy F1':>10} {'SUPPORT':>10}")
    print(f"  {'-'*50}")

# Per label
for label in NER_LABELS:
    spacy_f1   = spacy_results["per_label"][label]["f1"]
    support    = spacy_results["per_label"][label]["support"]

    if gliner_meta:
        gliner_f1 = gliner_meta.get("evaluation", {}).get("per_label_f1", {}).get(label, None)
        if gliner_f1 is not None:
            diff   = spacy_f1 - gliner_f1
            winner = "spaCy ↑" if diff > 0.05 else ("GLiNER ↑" if diff < -0.05 else "≈ egal")
            print(f"  {label:<25} {gliner_f1:>10.3f} {spacy_f1:>10.3f} {winner:>12}")
        else:
            print(f"  {label:<25} {'N/A':>10} {spacy_f1:>10.3f} {'N/A':>12}")
    else:
        if support > 0:
            print(f"  {label:<25} {spacy_f1:>10.3f} {support:>10}")
        else:
            print(f"  {label:<25} {'N/A':>10} {0:>10}")

# Global
print(f"  {'-'*62}" if gliner_meta else f"  {'-'*50}")
spacy_global = spacy_results["global"]["f1"]

if gliner_meta:
    gliner_global = gliner_meta.get("evaluation", {}).get("global_f1", None)
    if gliner_global:
        diff   = spacy_global - gliner_global
        winner = "spaCy ↑" if diff > 0.02 else ("GLiNER ↑" if diff < -0.02 else "≈ egal")
        print(f"  {'GLOBAL F1':<25} {gliner_global:>10.3f} {spacy_global:>10.3f} {winner:>12}")
else:
    print(f"  {'GLOBAL F1':<25} {spacy_global:>10.3f}")

# ─── Analiza arhitecturala ────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("ANALIZA COMPARATIVA — ASPECTE CHEIE PENTRU LUCRARE")
print("=" * 80)
print("""
  1. TAGSET FLEXIBIL
     GLiNER:  adaugi label nou → functioneaza instant (zero-shot)
     spaCy:   adaugi label nou → re-antrenare completa necesara

  2. VITEZA INFERENTA
     spaCy:   ~5-15ms per propozitie (linear head, simplu)
     GLiNER:  ~30-80ms per propozitie (encodare labels + span scoring)

  3. DATE NECESARE
     GLiNER:  performanta buna chiar si cu putine date (zero-shot fallback)
     spaCy:   necesita date suficiente per label pentru convergenta

  4. CONFIDENCE SCORE
     GLiNER:  expune scor de incredere per entitate (cosine similarity)
     spaCy:   nu expune confidence score nativ

  5. ENTITATI SUPRAPUSE (nested NER)
     GLiNER:  suporta nativ (span-based)
     spaCy:   nu suporta (transition-based parser, no overlap)
""")